# ProcureSight AI — Data Preparation

## 1. Purpose and Preparation Rules

Prepare an analysis dataset using the decisions documented in the data audit.

- Preserve the original CSV unchanged.
- Keep missing delivery dates and defect counts as unknown.
- Flag delivery dates that precede their order dates.
- Retain affected orders, but exclude invalid date intervals from calculations.
- Encode non-compliance as 1 and compliance as 0.
- Keep audit flags separate from future model-input decisions.

The meaning of Delivery_Date remains uncertain, so date intervals
must not yet be described as actual delivery lead times.

In [6]:
from pathlib import Path
import duckdb
import pandas as pd

# Locate the project whether the notebook runs from its folder or the root.
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "procurement_kpi.csv"

# Reload the source: this notebook can run independently of the audit.
raw_df = duckdb.read_csv(str(DATA_PATH)).df()
prepared_df = raw_df.copy()

# Record data-quality issues without inventing replacement values.
prepared_df["Missing_Delivery_Date"] = (
    prepared_df["Delivery_Date"].isna()
)
prepared_df["Missing_Defective_Units"] = (
    prepared_df["Defective_Units"].isna()
)
prepared_df["Invalid_Date_Sequence"] = (
    prepared_df["Delivery_Date"] < prepared_df["Order_Date"]
)

# Keep only non-negative, known intervals in this derived column.
date_interval = (
    prepared_df["Delivery_Date"] - prepared_df["Order_Date"]
).dt.days

prepared_df["Recorded_Date_Interval_Days"] = date_interval.where(
    date_interval >= 0
)

# Our risk target: 1 = non-compliant, 0 = compliant.
prepared_df["Non_Compliant"] = (
    prepared_df["Compliance"]
    .map({True: 0, False: 1})
    .astype("Int64")
)

preparation_summary = pd.Series({
    "Original orders": len(raw_df),
    "Prepared orders": len(prepared_df),
    "Missing delivery dates": prepared_df["Missing_Delivery_Date"].sum(),
    "Missing defect counts": prepared_df["Missing_Defective_Units"].sum(),
    "Invalid date sequences": prepared_df["Invalid_Date_Sequence"].sum(),
    "Usable recorded date intervals":
        prepared_df["Recorded_Date_Interval_Days"].notna().sum(),
    "Non-compliant orders": prepared_df["Non_Compliant"].sum(),
}, name="Count")

display(preparation_summary.to_frame())

,Count
Original orders,777
Prepared orders,777
Missing delivery dates,87
Missing defect counts,136
Invalid date sequences,1
Usable recorded date intervals,689
Non-compliant orders,137


## 2. Data Coverage Before Supplier Analysis

Coverage shows how many orders have usable data for each measurement.

- Recorded date intervals: exclude missing dates and invalid sequences.
- Defect counts: exclude missing values; unknown does not mean zero.
- Compliance labels: count records with a known target label.

These measures can have different denominators. Coverage must accompany
supplier KPIs so incomplete data is visible.

In [7]:
import plotly.express as px

total_orders = len(prepared_df)

coverage = pd.DataFrame({
    "Measure": [
        "Recorded date interval",
        "Defect count",
        "Compliance label",
    ],
    "Usable": [
        prepared_df["Recorded_Date_Interval_Days"].notna().sum(),
        prepared_df["Defective_Units"].notna().sum(),
        prepared_df["Non_Compliant"].notna().sum(),
    ],
})

coverage["Unavailable or invalid"] = total_orders - coverage["Usable"]
coverage["Coverage %"] = (
    coverage["Usable"] / total_orders * 100
).round(2)

display(coverage)

coverage_plot = coverage.melt(
    id_vars="Measure",
    value_vars=["Usable", "Unavailable or invalid"],
    var_name="Availability",
    value_name="Orders",
)

fig = px.bar(
    coverage_plot,
    x="Orders",
    y="Measure",
    color="Availability",
    orientation="h",
    text="Orders",
    barmode="stack",
    title="Data Coverage Across 777 Purchase Orders",
    color_discrete_map={
        "Usable": "#2563EB",
        "Unavailable or invalid": "#CBD5E1",
    },
    template="plotly_white",
)

fig.update_layout(
    height=380,
    xaxis_title="Number of orders",
    yaxis_title=None,
    legend_title_text=None,
)
fig.update_yaxes(autorange="reversed")
fig.update_traces(textposition="inside")

fig.show()

,Measure,Usable,Unavailable or invalid,Coverage %
0,Recorded date interval,689,88,88.67
1,Defect count,641,136,82.50
2,Compliance label,777,0,100.00


## 3. Data Coverage by Supplier

Compare the percentage of each supplier's orders with usable date
intervals and known defect counts.

Report both usable counts and percentages because suppliers have
different order volumes.

Coverage measures data availability, not supplier performance.
Missing records may affect the reliability of later comparisons.

In [8]:
# "size" counts all orders; "count" counts non-missing values.
supplier_coverage = (
    prepared_df.groupby("Supplier", dropna=False)
    .agg(
        Orders=("PO_ID", "size"),
        Usable_Intervals=("Recorded_Date_Interval_Days", "count"),
        Known_Defect_Counts=("Defective_Units", "count"),
    )
    .reset_index()
)

supplier_coverage["Date interval coverage %"] = (
    100 * supplier_coverage["Usable_Intervals"]
    / supplier_coverage["Orders"]
)

supplier_coverage["Defect count coverage %"] = (
    100 * supplier_coverage["Known_Defect_Counts"]
    / supplier_coverage["Orders"]
)

display(supplier_coverage.round(2))

# Convert the two percentage columns into rows for grouped bars.
supplier_coverage_plot = supplier_coverage.melt(
    id_vars=["Supplier", "Orders"],
    value_vars=[
        "Date interval coverage %",
        "Defect count coverage %",
    ],
    var_name="Measure",
    value_name="Coverage %",
)

fig = px.bar(
    supplier_coverage_plot,
    x="Supplier",
    y="Coverage %",
    color="Measure",
    barmode="group",
    text="Coverage %",
    hover_data={"Orders": True, "Coverage %": ":.2f"},
    title="Data Coverage by Supplier",
    color_discrete_map={
        "Date interval coverage %": "#2563EB",
        "Defect count coverage %": "#F97316",
    },
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{y:.1f}%",
    textposition="outside",
    cliponaxis=False,
)
fig.update_layout(
    height=450,
    xaxis_title=None,
    legend_title_text=None,
)
fig.update_yaxes(range=[0, 105], ticksuffix="%")

fig.show()

,Supplier,Orders,Usable_Intervals,Known_Defect_Counts,Date interval coverage %,Defect count coverage %
0,Alpha_Inc,141,116,113,82.27,80.14
1,Beta_Supplies,156,143,132,91.67,84.62
2,Delta_Logistics,171,151,127,88.30,74.27
3,Epsilon_Group,166,149,141,89.76,84.94
4,Gamma_Co,143,130,128,90.91,89.51


### Supplier Coverage Findings

- Delta_Logistics has the lowest defect-count coverage:
  127 of 171 orders (74.27%), leaving 44 orders without defect counts.
- Alpha_Inc has the lowest usable date-interval coverage:
  116 of 141 orders (82.27%).
- Gamma_Co has the highest defect-count coverage (89.51%).
- No supplier has complete coverage for either measurement.

Supplier KPIs must show their usable-record counts and coverage.
Missing defect counts must not be treated as zero.
These differences do not establish supplier performance rankings
or explain why records are missing.

## 4. Save the Prepared Dataset

Save the prepared data as Parquet, which preserves column types
such as dates, booleans, and numbers.

The original CSV remains unchanged. This output can be regenerated
by running this notebook.

In [9]:
# Create the output folder if it does not already exist.
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "procurement_prepared.parquet"

# Save the prepared table without the notebook's row index.
prepared_df.to_parquet(
    OUTPUT_PATH,
    engine="pyarrow",
    index=False,
)

# Confirm that saving and reloading preserves the table.
reloaded_df = pd.read_parquet(OUTPUT_PATH)
pd.testing.assert_frame_equal(prepared_df, reloaded_df)

print(f"Saved: {OUTPUT_PATH.name}")
print(f"Rows: {len(reloaded_df):,}")
print(f"Columns: {len(reloaded_df.columns)}")
print("Verification passed: reloaded data matches prepared data.")

Saved: procurement_prepared.parquet
Rows: 777
Columns: 16
Verification passed: reloaded data matches prepared data.
